### Model 1: NER PII/PHI Detector

In [6]:
# get the dataset
# scripts/download_ner_dataset.py
from datasets import load_dataset
import os

RAW_DIR = "/Users/Ravi/Documents/monroe_college/courses/sem_04/Applied Data Science Project/project/AI-Driven-Compliance-Guardian/data/raw/ner"
os.makedirs(RAW_DIR, exist_ok=True)

# Pick dataset (AI4Privacy has multilingual PII data)
dataset = load_dataset("ai4privacy/pii-masking-400k")

# Save splits as JSONL
dataset["train"].to_json(os.path.join(RAW_DIR, "train.jsonl"))
dataset["validation"].to_json(os.path.join(RAW_DIR, "validation.jsonl"))

print(f"Saved raw dataset to {RAW_DIR}")

Creating json from Arrow format: 100%|██████████| 82/82 [00:01<00:00, 74.23ba/s]

Saved raw dataset to /Users/Ravi/Documents/monroe_college/courses/sem_04/Applied Data Science Project/project/AI-Driven-Compliance-Guardian/data/raw/ner


In [8]:
# scripts/prepare_ner_dataset.py
import os, json
from tqdm import tqdm

RAW_DIR = "/Users/Ravi/Documents/monroe_college/courses/sem_04/Applied Data Science Project/project/AI-Driven-Compliance-Guardian/data/raw/ner"
PROC_DIR = "/Users/Ravi/Documents/monroe_college/courses/sem_04/Applied Data Science Project/project/AI-Driven-Compliance-Guardian/data/processed/ner"
os.makedirs(PROC_DIR, exist_ok=True)

def normalize(jsonl_path, out_path):
    out = []
    with open(jsonl_path, "r") as f:
        for line in tqdm(f, desc=f"Processing {jsonl_path}"):
            ex = json.loads(line)

            # pick correct text field
            text = ex.get("source_text") or ex.get("text") or ex.get("masked_text")

            # pick entities
            spans = ex.get("privacy_mask") or ex.get("spans") or []

            norm_spans = []
            for s in spans:
                try:
                    norm_spans.append({
                        "start": int(s["start"]),
                        "end": int(s["end"]),
                        "label": str(s.get("label") or "PII")
                    })
                except Exception as e:
                    print("⚠️ bad span:", s, e)

            out.append({"text": text, "spans": norm_spans})

    with open(out_path, "w") as f:
        for ex in out:
            f.write(json.dumps(ex) + "\n")

    print(f"✅ Wrote {len(out)} examples to {out_path}")

normalize(os.path.join(RAW_DIR, "train.jsonl"), os.path.join(PROC_DIR, "train.jsonl"))
normalize(os.path.join(RAW_DIR, "validation.jsonl"), os.path.join(PROC_DIR, "validation.jsonl"))

Processing /Users/Ravi/Documents/monroe_college/courses/sem_04/Applied Data Science Project/project/AI-Driven-Compliance-Guardian/data/raw/ner/train.jsonl: 325517it [00:03, 89959.00it/s] 


✅ Wrote 325517 examples to /Users/Ravi/Documents/monroe_college/courses/sem_04/Applied Data Science Project/project/AI-Driven-Compliance-Guardian/data/processed/ner/train.jsonl


Processing /Users/Ravi/Documents/monroe_college/courses/sem_04/Applied Data Science Project/project/AI-Driven-Compliance-Guardian/data/raw/ner/validation.jsonl: 81379it [00:00, 119558.79it/s]


✅ Wrote 81379 examples to /Users/Ravi/Documents/monroe_college/courses/sem_04/Applied Data Science Project/project/AI-Driven-Compliance-Guardian/data/processed/ner/validation.jsonl
